# Review and Score Data Agent Evaluation

Use this notebook after running baseline and final snapshots with `NB_Run_SDK_Evaluation.ipynb`. It records reviewed evidence, validates observations, calculates the deterministic eight-question scorecard, and exports submission artifacts.

Each question is worth four points in each phase: original answer, selected source, paraphrase consistency, and reviewed query logic. The maximum is **32 baseline** and **32 final**.

This notebook does not call the Data Agent SDK.

In [ ]:
# 1. Configure Evaluation Files

import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

DOMAIN_PROFILE = "water-utilities"
DOMAIN_SETTINGS = {
    "water-utilities": {"semantic_model": "WaterUtilitiesSemanticModel", "lakehouse": "WaterUtilitiesDemo"},
}
if DOMAIN_PROFILE not in DOMAIN_SETTINGS:
    raise ValueError(
        f"No scoring defaults exist for {DOMAIN_PROFILE!r}. Add its Fabric item names to DOMAIN_SETTINGS."
    )
KNOWN_FABRIC_ITEMS = set(DOMAIN_SETTINGS[DOMAIN_PROFILE].values())
DOMAIN_TOKEN = DOMAIN_PROFILE.replace("-", "_")

PARTICIPANT_ID = ""
RUN_MODE = "reviewed-scorecard"
EVALUATION_TIMESTAMP = datetime.now(timezone.utc).isoformat()
OUTPUT_DIRECTORY = Path(".")
OUTPUT_PREFIX = f"{DOMAIN_TOKEN}_data_agent_evaluation"
CSV_PATH = OUTPUT_DIRECTORY / f"{OUTPUT_PREFIX}_scorecard.csv"
JSON_PATH = OUTPUT_DIRECTORY / f"{OUTPUT_PREFIX}_report.json"
MAX_TOTAL = 32.0

REPOSITORY_OWNER = "hSushmithaShetty13"
REPOSITORY_NAME = "Water-Utilities"
REPOSITORY_REF = "main"
CHALLENGE_URL = (
    f"https://raw.githubusercontent.com/{REPOSITORY_OWNER}/"
    f"{REPOSITORY_NAME}/{REPOSITORY_REF}/evaluation/challenge/{DOMAIN_PROFILE}.json"
)

print("Domain profile:", DOMAIN_PROFILE)
print("Known Fabric items:", sorted(KNOWN_FABRIC_ITEMS))

In [ ]:
# 2. Load the Challenge Question and Paraphrase Pairs

challenge_response = requests.get(CHALLENGE_URL, timeout=60)
challenge_response.raise_for_status()
challenge_payload = challenge_response.json()

CHALLENGE = [
    {
        **item,
        "expected_answer": item["ground_truth_answer"],
        "expected_logic": item["ground_truth_sql"],
    }
    for item in challenge_payload["evaluation_queries"]
]
challenge_by_id = {item["id"]: item for item in CHALLENGE}
declared_query_count = challenge_payload["metadata"]["total_queries"]

assert declared_query_count == len(CHALLENGE)
assert len(challenge_by_id) == declared_query_count
print(
    f"Loaded {len(CHALLENGE)} questions from challenge version "
    f"{challenge_payload['metadata']['version']}."
 )

In [ ]:
# 3. Enter Baseline and Final Test Results

def blank_phase():
    return {
        "original_answer": "",
        "paraphrase_answer": "",
        "selected_source": "",
        "query_evidence": "",  # paste generated SQL/DAX or a concise copied run-step excerpt
        "logic_correct": None,  # set True only after table/measure, filters, and aggregation are verified
        "answer_observation": "",
        "logic_observation": "",
    }

OBSERVATIONS = [
    {"id": item["id"], "baseline": blank_phase(), "final": blank_phase()}
    for item in CHALLENGE
]

# Edit OBSERVATIONS above or update entries here before running the remaining cells.
# Example: OBSERVATIONS[0]["baseline"].update({"original_answer": 30, "paraphrase_answer": 30, "selected_source": DOMAIN_SETTINGS[DOMAIN_PROFILE]["semantic_model"], "query_evidence": "Copied SQL/DAX or run-step evidence", "logic_correct": True})
# Example: OBSERVATIONS[0]["final"].update({"original_answer": 30, "paraphrase_answer": 30, "selected_source": DOMAIN_SETTINGS[DOMAIN_PROFILE]["semantic_model"], "query_evidence": "Copied SQL/DAX or run-step evidence", "logic_correct": True})

In [ ]:
# 4. Validate Sources and Query Logic

def validate_observations(observations):

    issues = []

    ids = [entry.get("id") for entry in observations]

    if set(ids) != set(challenge_by_id) or len(ids) != len(challenge_by_id):

        expected_ids = ", ".join(challenge_by_id)

        issues.append(f"Observations must contain each challenge ID exactly once: {expected_ids}.")

    for entry in observations:

        test_id = entry.get("id", "unknown")

        for phase_name in ("baseline", "final"):

            phase = entry.get(phase_name, {})

            for field in ("original_answer", "paraphrase_answer", "selected_source"):

                if not str(phase.get(field, "")).strip():

                    issues.append(f"{test_id} {phase_name}: missing {field}.")

            source = str(phase.get("selected_source", "")).strip()

            if source and source not in KNOWN_FABRIC_ITEMS:

                issues.append(f"{test_id} {phase_name}: unknown Fabric item '{source}'.")

            if not str(phase.get("query_evidence", "")).strip():

                issues.append(f"{test_id} {phase_name}: add copied SQL/DAX evidence.")

            logic = phase.get("logic_correct")

            if logic not in (True, False, None):

                issues.append(f"{test_id} {phase_name}: logic_correct must be True, False, or None.")

            if logic is None:

                issues.append(f"{test_id} {phase_name}: complete the query-logic review.")

            if logic is True and not str(phase.get("query_evidence", "")).strip():

                issues.append(f"{test_id} {phase_name}: logic cannot be True without query evidence.")

    return issues



validation_issues = validate_observations(OBSERVATIONS)

validation_df = pd.DataFrame({"actionable_issue": validation_issues})

print(f"Validation found {len(validation_issues)} actionable issue(s).")

display(validation_df)


In [ ]:
# 5. Calculate Deterministic Scores

NUMBER_PATTERN = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?")



def answer_matches(actual, expected):

    if isinstance(actual, (int, float)) and not isinstance(actual, bool):

        return math.isclose(float(actual), float(expected), rel_tol=0.0, abs_tol=0.01)

    matches = NUMBER_PATTERN.findall(str(actual).strip())

    return len(matches) == 1 and math.isclose(

        float(matches[0].replace(",", "")), float(expected), rel_tol=0.0, abs_tol=0.01

    )



def score_phase(phase, expected):

    checks = {

        "answer": answer_matches(phase.get("original_answer"), expected["expected_answer"]),

        "source": str(phase.get("selected_source", "")).strip().casefold() == expected["expected_source"].casefold(),

        "consistency": answer_matches(phase.get("paraphrase_answer"), expected["expected_answer"]),

        "logic": phase.get("logic_correct") is True and bool(str(phase.get("query_evidence", "")).strip()),

    }

    return checks, float(sum(checks.values()))



scored_rows = []

for entry in OBSERVATIONS:

    expected = challenge_by_id[entry["id"]]

    baseline_checks, baseline_score = score_phase(entry["baseline"], expected)

    final_checks, final_score = score_phase(entry["final"], expected)

    scored_rows.append({

        "id": entry["id"], "question": expected["question"], "paraphrase": expected["paraphrase"],

        "expected_answer": expected["expected_answer"], "expected_source": expected["expected_source"],

        "baseline_score": baseline_score, "final_score": final_score, "change": final_score - baseline_score,

        "baseline_source": entry["baseline"].get("selected_source"), "final_source": entry["final"].get("selected_source"),

        "baseline_logic": entry["baseline"].get("logic_correct"), "final_logic": entry["final"].get("logic_correct"),

        "baseline_checks": baseline_checks, "final_checks": final_checks,

        "baseline_query_evidence": entry["baseline"].get("query_evidence", ""),

        "final_query_evidence": entry["final"].get("query_evidence", ""),

    })



scorecard_df = pd.DataFrame(scored_rows)

baseline_total = float(scorecard_df["baseline_score"].sum())

final_total = float(scorecard_df["final_score"].sum())

assert 0.0 <= baseline_total <= MAX_TOTAL and 0.0 <= final_total <= MAX_TOTAL


In [ ]:
# 6. Build the Per-Question Scorecard

scorecard_df["validation_status"] = [

    "Ready" if row["baseline_score"] == 4 and row["final_score"] == 4 else "Review"

    for _, row in scorecard_df.iterrows()

]

scorecard_df["actionable_issue"] = [

    ", ".join(

        f"{phase} {name}"

        for phase in ("baseline", "final")

        for name, passed in row[f"{phase}_checks"].items()

        if not passed

    )

    for _, row in scorecard_df.iterrows()

]

display(scorecard_df[[

    "id", "question", "baseline_score", "final_score", "change",

    "baseline_source", "final_source", "baseline_logic", "final_logic",

    "validation_status", "actionable_issue",

]])


In [ ]:
# 7. Compare Baseline and Final Totals

improvement = final_total - baseline_total

percentage_improvement = None if baseline_total == 0 else 100.0 * improvement / baseline_total

regressions = scorecard_df.loc[scorecard_df["change"] < 0, "id"].tolist()

summary = {

    "baseline_score": baseline_total, "baseline_max": MAX_TOTAL,

    "final_score": final_total, "final_max": MAX_TOTAL,

    "absolute_improvement": improvement,

    "percentage_improvement": percentage_improvement,

    "question_regressions": regressions,

    "actionable_issue_count": len(validation_issues),

}

display(pd.DataFrame([summary]))


In [ ]:
# 8. Export CSV and JSON Evidence

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

export_df = scorecard_df.copy()

for column in ("baseline_checks", "final_checks"):

    export_df[column] = export_df[column].map(json.dumps)

export_df.to_csv(CSV_PATH, index=False)



report = {

    "metadata": {

        "participant_id": PARTICIPANT_ID,

        "evaluation_timestamp": EVALUATION_TIMESTAMP,

        "run_mode": RUN_MODE,

    },

    "summary": summary,

    "validation_issues": validation_issues,

    "observations": OBSERVATIONS,

    "scorecard": scored_rows,

}

JSON_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("CSV:", CSV_PATH.resolve())

print("JSON:", JSON_PATH.resolve())


In [ ]:
# 9. Validate Submission Artifacts

reloaded_csv = pd.read_csv(CSV_PATH)

reloaded_report = json.loads(JSON_PATH.read_text(encoding="utf-8"))

required_csv_columns = {"id", "baseline_score", "final_score", "baseline_query_evidence", "final_query_evidence"}

artifact_checks = {

    "CSV exists": CSV_PATH.is_file(),

    "JSON exists": JSON_PATH.is_file(),

    "CSV schema valid": required_csv_columns.issubset(reloaded_csv.columns),

    "Challenge question pairs present": set(reloaded_csv["id"]) == set(challenge_by_id),

    "Baseline total matches": math.isclose(float(reloaded_csv["baseline_score"].sum()), reloaded_report["summary"]["baseline_score"]),

    "Final total matches": math.isclose(float(reloaded_csv["final_score"].sum()), reloaded_report["summary"]["final_score"]),

    "Query evidence complete": all(

        str(phase.get("query_evidence", "")).strip()

        for item in OBSERVATIONS for phase in (item["baseline"], item["final"])

    ),

    "Screenshots or copied run-step evidence attached": False,  # set True before submission

}

submission_checklist_df = pd.DataFrame(

    [{"check": name, "complete": complete} for name, complete in artifact_checks.items()]

)

display(submission_checklist_df)
